Sesion05 - Manejo de errores

Definir el esquema

In [0]:
from pyspark.sql.types import StructType, StringType, IntegerType, FloatType

estudiantes_schema = (
    StructType()
    .add("Student_ID", StringType())
    .add("Semester_ID", StringType())
    .add("Age", IntegerType())
    .add("Gender", StringType())
    .add("Region_Type", StringType())
    .add("Major_Subject", StringType())
    .add("Final_Exam_Score", FloatType())
    .add("_rescued_data", StringType())
)

print("Esquema definido correctamente")

Leer los JSON con detección de errores

In [0]:
from pyspark.sql.functions import col

df_estudiantes = (
    spark.read
    .format("json")
    .schema(estudiantes_schema)
    .option("columnNameOfCorruptRecord", "_rescued_data")
    .option("badRecordsPath", "/Volumes/proyecto_estudiantes1/bronze/volumen1/input/json/bad_records/")
    .load("/Volumes/proyecto_estudiantes1/bronze/volumen1/input/json/")
    .withColumn("archivo_origen", col("_metadata.file_path"))
)

print("Datos cargados correctamente")
df_estudiantes.printSchema()

Ver los datos cargados

In [0]:
display(df_estudiantes)

Separar datos válidos e inválidos

In [0]:
from pyspark.sql.functions import col

try:
    df_validos = df_estudiantes.filter(col("_rescued_data").isNull())
    df_invalidos = df_estudiantes.filter(col("_rescued_data").isNotNull())

    df_validos.write.format("delta").mode("overwrite").saveAsTable(
        "proyecto_estudiantes1.silver.estudiantes_validos"
    )
    df_invalidos.write.format("delta").mode("overwrite").saveAsTable(
        "proyecto_estudiantes1.silver.estudiantes_invalidos"
    )

    print("Escritura realizada con éxito.")

except Exception as e:
    print(f"Error durante la escritura: {str(e)}")

Validar cuántos válidos e inválidos hay

In [0]:
%sql
SELECT 'Validos' AS tipo, COUNT(*) AS cantidad FROM proyecto_estudiantes1.silver.estudiantes_validos
UNION ALL
SELECT 'Invalidos' AS tipo, COUNT(*) AS cantidad FROM proyecto_estudiantes1.silver.estudiantes_invalidos;

Ver los registros inválidos

In [0]:
%sql
SELECT * FROM proyecto_estudiantes1.silver.estudiantes_validos;